# SafeSynth — RT-DETRv2 四組對照訓練

**跑之前確認兩件事：**

1. Runtime 是 **L4 GPU**（執行階段 → 變更執行階段類型）
2. Drive 有 `sdg-portfolio/02-safesynth-ppe/safesynth_train_data.zip`

然後「執行階段 → 全部執行」，四組依序跑完，預估 **4–5 小時**。
斷線就重新全部執行一次，會自動從 checkpoint 接續。

**不需要任何 token。** 模型是公開的，這本 notebook 不讀 Secrets、不上傳任何東西。

## 1. 確認 GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "沒有 GPU。執行階段 → 變更執行階段類型 → L4 GPU"
_name = torch.cuda.get_device_name(0)
_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{_name}  {_total_gb:.0f} GB  bf16={torch.cuda.is_bf16_supported()}")
if not torch.cuda.is_bf16_supported():
    print(
        "WARNING: 這張卡不支援 bf16（大概是 T4），會退回 fp16。\n"
        "         這個模型在 fp16 下比較容易出現 NaN loss，建議改用 L4。"
    )

## 2. 安裝相依套件

In [ ]:
%pip install -q "transformers>=5.14.1" albumentations pycocotools accelerate

import transformers

print("transformers", transformers.__version__)

## 3. 掛載 Drive，解壓到本機磁碟

**不直接從 Drive 讀圖訓練**（TRAIN-08）——Drive 的隨機讀取延遲會讓 GPU 大部分時間在等 I/O。

In [ ]:
import pathlib
import shutil
import time

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = '/content/drive/MyDrive/sdg-portfolio/02-safesynth-ppe'
ARCHIVE = 'safesynth_train_data.zip'

archive = pathlib.Path(DRIVE_DIR) / ARCHIVE
assert archive.is_file(), f"找不到 {archive}，確認 Drive 路徑與檔名"
print(f"{archive.stat().st_size / 1e9:.2f} GB")

DATA = pathlib.Path("/content/data")
if not (DATA / "real" / "coco_all.json").is_file():
    DATA.mkdir(parents=True, exist_ok=True)
    _start = time.time()
    shutil.unpack_archive(str(archive), str(DATA), "zip")
    print(f"unpacked in {time.time() - _start:.0f}s")
else:
    print("already unpacked")

n_real = len(list((DATA / "real" / "images").glob("*.png")))
n_syn = len(list((DATA / "synthetic" / "images").glob("*.png")))
print(f"real={n_real}  synthetic={n_syn}")
assert n_real == 4256, "真實影像張數不符，資料包可能不完整"
assert n_syn == 6152, "合成影像張數不符，資料包可能不完整"

## 4. 寫入訓練模組

這是本機 `uv run pytest` 測過的同一份程式碼，內嵌於此以免 Colab 需要 clone 一個還不存在的 GitHub repo。

In [ ]:
import pathlib
import sys

# The modules import each other as `src.training.x`, so the package layout has
# to exist before any of them is written.
ROOT = pathlib.Path("/content/safesynth")
PKG = ROOT / "src" / "training"
PKG.mkdir(parents=True, exist_ok=True)
(ROOT / "src" / "__init__.py").write_text("", encoding="utf-8")
(PKG / "__init__.py").write_text("", encoding="utf-8")

MODULE_SOURCE = {}

MODULE_SOURCE['arms'] = r'''"""Which images each of the four arms trains on (TRAIN-03..07).

This module is the single source of truth for arm composition, and it is a
module rather than notebook cells for one reason: notebooks cannot be tested,
and every rule here is a rule whose violation is silent. An arm that quietly
trains on a different set of real images, or synthetic arms of unequal size,
produces a clean-looking result table that means nothing.

`scripts/build_training_notebook.py` embeds this file into the Colab notebook,
so the code that runs on Colab is the code the tests ran against.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

ARMS = ("real_only", "standard_aug", "unfiltered_syn", "filtered_syn")
SYNTHETIC_SUBSET = {
    "real_only": None,
    "standard_aug": None,
    "unfiltered_syn": "unfiltered",
    "filtered_syn": "filtered",
}
# The arms differ in augmentation only between real_only and the rest: TRAIN-05
# requires standard_aug's photometric range to bracket what synthetic images
# carry, and the synthetic arms must reuse that same augmentation or "synthetic
# helps" becomes confounded with "different augmentation".
AUGMENTATION_PROFILE = {
    "real_only": "real_only",
    "standard_aug": "standard_aug",
    "unfiltered_syn": "standard_aug",
    "filtered_syn": "standard_aug",
}


class ArmCompositionError(RuntimeError):
    """Raised when an arm would train on the wrong data."""


@dataclass(frozen=True)
class ArmComposition:
    arm: str
    real_train: tuple[str, ...]
    real_val: tuple[str, ...]
    synthetic: tuple[str, ...]
    augmentation_profile: str
    real_train_digest: str = field(default="")

    @property
    def n_train_images(self) -> int:
        return len(self.real_train) + len(self.synthetic)

    def summary(self) -> dict[str, Any]:
        return {
            "arm": self.arm,
            "n_real_train": len(self.real_train),
            "n_real_val": len(self.real_val),
            "n_synthetic": len(self.synthetic),
            "n_train_total": self.n_train_images,
            "augmentation_profile": self.augmentation_profile,
            "real_train_digest": self.real_train_digest,
        }


def digest_names(names: Sequence[str]) -> str:
    """Order-independent digest, so TRAIN-04 compares sets not orderings."""

    payload = "\n".join(sorted(names)).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _coco_image_names(path: Path) -> tuple[str, ...]:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    return tuple(sorted(image["file_name"].split("/")[-1] for image in payload["images"]))


def split_real_images(manifest_path: Path) -> dict[str, tuple[str, ...]]:
    """Read the frozen split. Test is returned so callers can assert against it."""

    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
    entries = manifest["images"] if isinstance(manifest, dict) else manifest
    grouped: dict[str, list[str]] = {}
    for entry in entries:
        name = str(entry["file_name"]).split("/")[-1]
        grouped.setdefault(str(entry["split"]), []).append(name)
    return {split: tuple(sorted(names)) for split, names in grouped.items()}


def build_arm(
    arm: str,
    *,
    real_splits: Mapping[str, Sequence[str]],
    synthetic_annotations: Mapping[str, Path],
) -> ArmComposition:
    if arm not in ARMS:
        raise ArmCompositionError(f"Unknown arm {arm!r}; expected one of {ARMS}")

    real_train = tuple(sorted(real_splits["train"]))
    real_val = tuple(sorted(real_splits["val"]))
    if not real_train or not real_val:
        raise ArmCompositionError("Frozen split yielded an empty train or val set")

    subset = SYNTHETIC_SUBSET[arm]
    synthetic: tuple[str, ...] = ()
    if subset is not None:
        path = synthetic_annotations.get(subset)
        if path is None:
            raise ArmCompositionError(f"Arm {arm!r} needs the {subset!r} annotations")
        synthetic = _coco_image_names(Path(path))
        if not synthetic:
            raise ArmCompositionError(f"{subset!r} annotations contain no images")

    return ArmComposition(
        arm=arm,
        real_train=real_train,
        real_val=real_val,
        synthetic=synthetic,
        augmentation_profile=AUGMENTATION_PROFILE[arm],
        real_train_digest=digest_names(real_train),
    )


def build_all_arms(
    *,
    manifest_path: Path,
    synthetic_annotations: Mapping[str, Path],
) -> dict[str, ArmComposition]:
    real_splits = split_real_images(manifest_path)
    compositions = {
        arm: build_arm(
            arm, real_splits=real_splits, synthetic_annotations=synthetic_annotations
        )
        for arm in ARMS
    }
    assert_arm_invariants(compositions, real_splits=real_splits)
    return compositions


def assert_arm_invariants(
    compositions: Mapping[str, ArmComposition],
    *,
    real_splits: Mapping[str, Sequence[str]],
) -> None:
    """Every rule here fails silently if unchecked, so all of them crash instead."""

    missing = set(ARMS) - set(compositions)
    if missing:
        raise ArmCompositionError(f"Missing arms: {sorted(missing)}")

    # TRAIN-23 — checked FIRST because it is the most severe failure and the one
    # least likely to be noticed. A leak that touches all four arms equally would
    # sail past the digest comparison below, and a leak in one arm would
    # otherwise be reported as a mere "arms disagree", burying the real problem.
    test_names = set(real_splits.get("test", ()))
    if test_names:
        for arm, comp in compositions.items():
            leaked = test_names & (set(comp.real_train) | set(comp.real_val))
            if leaked:
                raise ArmCompositionError(
                    f"Arm {arm!r} would train or validate on {len(leaked)} Test images"
                )

    # TRAIN-04 — all four arms must consume exactly the same real images.
    digests = {arm: comp.real_train_digest for arm, comp in compositions.items()}
    if len(set(digests.values())) != 1:
        raise ArmCompositionError(
            f"Arms disagree on their real training images: {digests}"
        )

    # TRAIN-06 — the two synthetic arms must be the same size, or the ablation
    # confounds "more data" with "better data".
    unfiltered = len(compositions["unfiltered_syn"].synthetic)
    filtered = len(compositions["filtered_syn"].synthetic)
    if unfiltered != filtered:
        raise ArmCompositionError(
            f"Synthetic arms are not size matched: unfiltered={unfiltered} "
            f"filtered={filtered}"
        )

    # The baseline arms must carry no synthetic data at all.
    for arm in ("real_only", "standard_aug"):
        if compositions[arm].synthetic:
            raise ArmCompositionError(f"Arm {arm!r} must not receive synthetic images")

    # Real val must be identical across arms too; a differing val set makes the
    # headline numbers incomparable while looking perfectly normal.
    val_digests = {arm: digest_names(comp.real_val) for arm, comp in compositions.items()}
    if len(set(val_digests.values())) != 1:
        raise ArmCompositionError(f"Arms disagree on their validation images: {val_digests}")


def equal_step_budget(
    compositions: Mapping[str, ArmComposition],
    *,
    reference_arm: str,
    reference_epochs: int,
    batch_size: int,
) -> dict[str, dict[str, Any]]:
    """Give every arm the same optimizer steps (TRAIN-07, `equal_steps`).

    The three quantities TRAIN-07 asks to align - optimizer steps, batch size and
    real-image exposure - cannot all hold once the datasets differ in size, so
    one has to give. Fixing steps controls compute, which answers the sharper
    objection ("it only won because it trained longer"). The cost is that the
    synthetic arms see each real image fewer times, so that number is returned
    per arm and belongs in the report rather than in a footnote.
    """

    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    reference = compositions[reference_arm]
    steps_per_epoch = max(1, reference.n_train_images // batch_size)
    total_steps = steps_per_epoch * reference_epochs

    plan: dict[str, dict[str, Any]] = {}
    for arm, comp in compositions.items():
        arm_steps_per_epoch = max(1, comp.n_train_images // batch_size)
        epochs = total_steps / arm_steps_per_epoch
        real_fraction = len(comp.real_train) / comp.n_train_images
        plan[arm] = {
            "total_steps": total_steps,
            "steps_per_epoch": arm_steps_per_epoch,
            "epochs": round(epochs, 3),
            "n_train_images": comp.n_train_images,
            # What the report has to state plainly: how often each real image is
            # actually seen once the step budget is held fixed.
            "real_image_exposures": round(
                total_steps * batch_size * real_fraction / len(comp.real_train), 2
            ),
        }
    return plan
'''
MODULE_SOURCE['data'] = r'''"""Dataset, augmentation and collation for RT-DETRv2 fine-tuning.

The coordinate convention is the whole story here. ADR-014 verified by execution
that the HF image processor converts COCO [x, y, w, h] absolute into normalized
cxcywh, which is exactly what the loss wants. So:

    albumentations stays in COCO absolute xywh from end to end
    the image processor performs the one and only conversion
    nothing in between ever touches cxcywh

Doing that conversion by hand, or twice, does not raise. It produces a model
that trains to a plausible-looking loss curve and detects nothing.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

CLASS_NAMES = ("helmet", "head", "person")
# TRAIN's "category id remapped to 0..K-1" rule. A non-contiguous COCO
# category_id is the classic cause of "loss falls nicely but mAP is near zero".
NAME_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}


class AnnotationConventionError(RuntimeError):
    """Raised when boxes are not in the convention this pipeline requires."""


@dataclass(frozen=True)
class Sample:
    image_path: Path
    image_id: int
    boxes_xywh: tuple[tuple[float, float, float, float], ...]
    class_indices: tuple[int, ...]
    is_synthetic: bool
    # Needed to rescale predictions back to annotation coordinates. Not all
    # images are exactly 416x416 - some synthetic outputs are 415 wide after
    # reflected-padding normalization - so this is read per image rather than
    # assumed.
    width: int = 416
    height: int = 416


def load_coco_samples(
    annotations_path: Path,
    images_root: Path,
    *,
    keep_names: Sequence[str] | None = None,
    is_synthetic: bool = False,
) -> list[Sample]:
    """Read a COCO file into samples, remapping category ids to 0..K-1."""

    import json

    payload = json.loads(Path(annotations_path).read_text(encoding="utf-8"))
    id_to_name = {int(c["id"]): str(c["name"]) for c in payload["categories"]}
    unknown = set(id_to_name.values()) - set(CLASS_NAMES)
    if unknown:
        raise AnnotationConventionError(f"Unexpected categories: {sorted(unknown)}")

    allow = set(keep_names) if keep_names is not None else None
    images = {}
    for image in payload["images"]:
        name = image["file_name"].split("/")[-1]
        if allow is None or name in allow:
            images[int(image["id"])] = (
                name,
                int(image.get("width", 416)),
                int(image.get("height", 416)),
            )

    grouped: dict[int, list[dict[str, Any]]] = {image_id: [] for image_id in images}
    for annotation in payload["annotations"]:
        image_id = int(annotation["image_id"])
        if image_id in grouped:
            grouped[image_id].append(annotation)

    samples: list[Sample] = []
    for image_id, (name, width, height) in sorted(
        images.items(), key=lambda item: item[1][0]
    ):
        boxes: list[tuple[float, float, float, float]] = []
        classes: list[int] = []
        for annotation in sorted(grouped[image_id], key=lambda a: int(a["id"])):
            x, y, w, h = (float(v) for v in annotation["bbox"])
            if w <= 0 or h <= 0:
                continue
            boxes.append((x, y, w, h))
            classes.append(NAME_TO_INDEX[id_to_name[int(annotation["category_id"])]])
        samples.append(
            Sample(
                image_path=Path(images_root) / name,
                image_id=image_id,
                boxes_xywh=tuple(boxes),
                class_indices=tuple(classes),
                is_synthetic=is_synthetic,
                width=width,
                height=height,
            )
        )
    return samples


def build_transform(profile: str, config: Mapping[str, Any]):
    """Albumentations pipeline in COCO absolute xywh (TRAIN-05).

    `real_only` gets nothing at all. `standard_aug` gets geometry plus a
    photometric block whose ranges track configs/compose.yaml, so the baseline
    has seen the same kind of degradation the synthetic images carry.
    """

    import albumentations as A

    settings = config["augmentation"]
    bbox_params = A.BboxParams(
        format=settings["bbox_format"],
        label_fields=["class_indices"],
        clip=bool(settings["bbox_clip"]),
        min_area=float(settings["bbox_min_area"]),
    )

    if profile == "real_only":
        return A.Compose([], bbox_params=bbox_params)

    arm = settings["standard_aug"]
    gamma_low, gamma_high = (float(v) for v in arm["gamma_range"])
    noise_low, noise_high = (float(v) for v in arm["gauss_noise_sigma"])
    jpeg_low, jpeg_high = (int(v) for v in arm["jpeg_quality"])
    return A.Compose(
        [
            A.HorizontalFlip(p=float(arm["horizontal_flip"])),
            A.Perspective(p=float(arm["perspective"])),
            A.RandomBrightnessContrast(p=float(arm["random_brightness_contrast"])),
            A.HueSaturationValue(p=float(arm["hue_saturation_value"])),
            # RandomGamma takes PERCENT. Passing 1.81 here is a silent no-op,
            # which is why the config comment shouts about it.
            A.RandomGamma(gamma_limit=(gamma_low * 100, gamma_high * 100), p=0.5),
            A.GaussNoise(std_range=(noise_low / 255.0, noise_high / 255.0), p=0.3),
            A.MotionBlur(blur_limit=int(arm["blur_limit"]), p=float(arm["motion_blur"])),
            A.ImageCompression(quality_range=(jpeg_low, jpeg_high), p=0.2),
        ],
        bbox_params=bbox_params,
    )


class DetectionDataset(Dataset):
    """Applies augmentation in COCO xywh, then hands the processor the conversion."""

    def __init__(self, samples: Sequence[Sample], processor, transform=None) -> None:
        self.samples = list(samples)
        self.processor = processor
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict[str, Any]:
        sample = self.samples[index]
        image = np.asarray(Image.open(sample.image_path).convert("RGB"))
        boxes = [list(box) for box in sample.boxes_xywh]
        classes = list(sample.class_indices)

        if self.transform is not None:
            augmented = self.transform(
                image=image, bboxes=boxes, class_indices=classes
            )
            image = augmented["image"]
            boxes = [list(box) for box in augmented["bboxes"]]
            classes = list(augmented["class_indices"])

        annotations = {
            "image_id": sample.image_id,
            "annotations": [
                {
                    "image_id": sample.image_id,
                    "category_id": int(category),
                    "bbox": [float(v) for v in box],
                    "area": float(box[2]) * float(box[3]),
                    "iscrowd": 0,
                }
                for box, category in zip(boxes, classes, strict=True)
            ],
        }
        encoded = self.processor(
            images=image, annotations=annotations, return_tensors="pt"
        )
        return {
            "pixel_values": encoded["pixel_values"][0],
            "labels": encoded["labels"][0],
        }


def collate(batch: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """`labels` stays a list, never a stacked tensor.

    Per-image label dicts have different lengths. Stacking them is what
    `eval_do_concat_batches=False` exists to prevent on the eval side; the
    collator has to respect the same shape on the train side.
    """

    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": [item["labels"] for item in batch],
    }


def assert_eval_batch_remainder_is_safe(n_val: int, batch_size: int) -> None:
    """A final eval batch of exactly one crashes collect_targets.

    image_size collapses from tensor([H, W]) to tensor([H]) and unpacking raises
    "not enough values to unpack (expected 2, got 1)". Confirmed upstream; one
    assertion at startup is cheaper than discovering it after an epoch of
    training on Colab.
    """

    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    if n_val % batch_size == 1:
        raise AnnotationConventionError(
            f"Validation size {n_val} leaves a remainder of exactly 1 at eval batch "
            f"size {batch_size}, which crashes collect_targets. Change the batch size "
            f"or move one image between splits."
        )
'''
MODULE_SOURCE['trainer'] = r'''"""Trainer subclass and TrainingArguments assembly.

Two things live here that Trainer does not give you:

1. Per-parameter-group learning rates. The upstream RT-DETRv2 recipe puts the
   backbone on 0.1x LR. That is strictly better than freezing it on a
   domain-shifted dataset like this one, and Trainer has no per-group LR, so
   create_optimizer has to be overridden.
2. The TrainingArguments that are not optional. Four of them are defaults that
   are wrong for detection, and getting them wrong does not raise.
"""

from __future__ import annotations

from collections.abc import Mapping
from typing import Any

import torch
from transformers import Trainer, TrainingArguments


def build_parameter_groups(
    model: torch.nn.Module,
    *,
    learning_rate: float,
    backbone_lr_multiplier: float,
    weight_decay: float,
    no_decay_on_norm_and_bias: bool = True,
) -> list[dict[str, Any]]:
    """Three groups: backbone at 0.1x, decayable at 1x, norm/bias at 1x with no decay."""

    backbone, decay, no_decay = [], [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if "backbone" in name:
            backbone.append(parameter)
        elif no_decay_on_norm_and_bias and (
            name.endswith(".bias") or parameter.ndim == 1
        ):
            no_decay.append(parameter)
        else:
            decay.append(parameter)

    groups = [
        {
            "params": backbone,
            "lr": learning_rate * backbone_lr_multiplier,
            "weight_decay": weight_decay,
            "name": "backbone",
        },
        {
            "params": decay,
            "lr": learning_rate,
            "weight_decay": weight_decay,
            "name": "decay",
        },
        {
            "params": no_decay,
            "lr": learning_rate,
            "weight_decay": 0.0,
            "name": "no_decay",
        },
    ]
    return [group for group in groups if group["params"]]


class RTDetrTrainer(Trainer):
    """Trainer with the upstream three-group optimizer."""

    def __init__(self, *args, optimizer_config: Mapping[str, Any] | None = None, **kwargs):
        self._optimizer_config = dict(optimizer_config or {})
        super().__init__(*args, **kwargs)

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer
        config = self._optimizer_config
        groups = build_parameter_groups(
            self.model,
            learning_rate=float(config.get("learning_rate", self.args.learning_rate)),
            backbone_lr_multiplier=float(config.get("backbone_lr_multiplier", 0.1)),
            weight_decay=float(config.get("weight_decay", self.args.weight_decay)),
            no_decay_on_norm_and_bias=bool(
                config.get("no_decay_on_norm_and_bias", True)
            ),
        )
        betas = tuple(float(b) for b in config.get("betas", (0.9, 0.999)))
        self.optimizer = torch.optim.AdamW(groups, betas=betas)
        return self.optimizer


# The four settings whose Trainer defaults are wrong for detection, and whose
# wrongness is silent rather than loud.
MANDATORY_TRAINING_ARGUMENTS = {
    # Trainer would otherwise concatenate ragged per-image label lists.
    "eval_do_concat_batches": False,
    # Trainer would strip the columns the dataset transform needs.
    "remove_unused_columns": False,
    # Upstream clip_max_norm is 0.1, not Trainer's default 1.0. The usual NaN
    # loss reports trace back to too-high LR or too-loose clipping.
    "max_grad_norm": 0.1,
}


def build_training_arguments(
    *,
    output_dir: str,
    config: Mapping[str, Any],
    total_steps: int,
    seed: int,
    use_bf16: bool,
    dataloader_num_workers: int,
) -> TrainingArguments:
    run = config["run"]
    schedule = config["schedule"]
    optimizer = config["optimizer"]

    arguments = TrainingArguments(
        output_dir=output_dir,
        seed=seed,
        max_steps=int(total_steps),
        per_device_train_batch_size=int(run["per_device_train_batch_size"]),
        per_device_eval_batch_size=int(run["per_device_eval_batch_size"]),
        learning_rate=float(optimizer["learning_rate"]),
        weight_decay=float(optimizer["weight_decay"]),
        lr_scheduler_type=str(schedule["lr_scheduler_type"]),
        warmup_steps=int(schedule["warmup_steps"]),
        max_grad_norm=float(schedule["max_grad_norm"]),
        bf16=bool(use_bf16),
        fp16=not bool(use_bf16),
        dataloader_num_workers=int(dataloader_num_workers),
        eval_strategy=str(run["eval_strategy"]),
        save_strategy=str(run["save_strategy"]),
        save_total_limit=int(run["save_total_limit"]),
        load_best_model_at_end=bool(run["load_best_model_at_end"]),
        metric_for_best_model=str(run["metric_for_best_model"]),
        greater_is_better=bool(run["greater_is_better"]),
        remove_unused_columns=MANDATORY_TRAINING_ARGUMENTS["remove_unused_columns"],
        eval_do_concat_batches=MANDATORY_TRAINING_ARGUMENTS["eval_do_concat_batches"],
        logging_steps=50,
        report_to=[],
    )
    assert_mandatory_arguments(arguments)
    return arguments


def assert_mandatory_arguments(arguments: TrainingArguments) -> None:
    """Fail loudly rather than train for hours on a silently wrong setting."""

    wrong = {
        key: getattr(arguments, key)
        for key, expected in MANDATORY_TRAINING_ARGUMENTS.items()
        if getattr(arguments, key) != expected
    }
    if wrong:
        raise RuntimeError(
            f"TrainingArguments deviate from the mandatory detection settings: {wrong}. "
            f"Expected {MANDATORY_TRAINING_ARGUMENTS}."
        )


def find_resumable_checkpoint(output_dir) -> str | None:
    """Colab disconnects; resuming is not an optional feature (TRAIN-10)."""

    from pathlib import Path

    directory = Path(output_dir)
    if not directory.is_dir():
        return None
    checkpoints = [
        path
        for path in directory.iterdir()
        if path.is_dir() and path.name.startswith("checkpoint-")
    ]
    if not checkpoints:
        return None

    def step_of(path) -> int:
        try:
            return int(path.name.split("-")[-1])
        except ValueError:
            return -1

    latest = max(checkpoints, key=step_of)
    return str(latest) if step_of(latest) >= 0 else None
'''
MODULE_SOURCE['metrics'] = r'''"""COCO mAP for the Trainer eval loop, via pycocotools.

pycocotools rather than torchmetrics on purpose: it is already a dependency, the
compositor already self-evaluates with COCOeval, and Colab then needs no extra
install that could resolve to a different version than the local smoke test ran
against. Same evaluator locally and remotely is worth more than convenience.

AP_small is one of the two headline metrics for this project, so it is pulled
out by name rather than left inside the twelve-number COCOeval dump.
"""

from __future__ import annotations

import contextlib
import io
from collections.abc import Sequence
from typing import Any

import numpy as np

# COCOeval's stats vector is positional and undocumented at the call site.
COCO_STAT_NAMES = (
    "map",           # AP @ IoU 0.50:0.95, all areas
    "map_50",
    "map_75",
    "map_small",     # headline metric #1
    "map_medium",
    "map_large",
    "mar_1",
    "mar_10",
    "mar_100",
    "mar_small",
    "mar_medium",
    "mar_large",
)


def build_coco_ground_truth(
    samples: Sequence[Any], class_names: Sequence[str]
) -> dict[str, Any]:
    """A COCO dict from the same Sample objects the dataset iterates."""

    images, annotations = [], []
    annotation_id = 1
    for sample in samples:
        images.append({"id": int(sample.image_id), "width": 0, "height": 0})
        for box, category in zip(sample.boxes_xywh, sample.class_indices, strict=True):
            x, y, w, h = (float(v) for v in box)
            annotations.append(
                {
                    "id": annotation_id,
                    "image_id": int(sample.image_id),
                    "category_id": int(category),
                    "bbox": [x, y, w, h],
                    "area": w * h,
                    "iscrowd": 0,
                }
            )
            annotation_id += 1
    return {
        "images": images,
        "annotations": annotations,
        "categories": [
            {"id": index, "name": name} for index, name in enumerate(class_names)
        ],
    }


def evaluate_detections(
    ground_truth: dict[str, Any], detections: Sequence[dict[str, Any]]
) -> dict[str, float]:
    """Run COCOeval and return the stats vector under readable names."""

    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    if not detections:
        # An untrained model legitimately predicts nothing above threshold at
        # step 1. Returning zeros keeps the smoke test meaningful instead of
        # crashing inside pycocotools on an empty result set.
        return dict.fromkeys(COCO_STAT_NAMES, 0.0)

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO()
        coco_gt.dataset = ground_truth
        coco_gt.createIndex()
        coco_dt = coco_gt.loadRes([dict(d) for d in detections])
        evaluator = COCOeval(coco_gt, coco_dt, iouType="bbox")
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    stats = np.asarray(evaluator.stats, dtype=float)
    return {
        name: float(stats[index]) if index < len(stats) and np.isfinite(stats[index]) else 0.0
        for index, name in enumerate(COCO_STAT_NAMES)
    }


class EvalStructureError(RuntimeError):
    """Raised when the eval output does not contain identifiable logits/boxes."""


def extract_logits_and_boxes(batch: Any, *, num_labels: int):
    """Find logits and pred_boxes inside one eval batch, BY SHAPE not by index.

    With `eval_do_concat_batches=False`, Trainer hands compute_metrics a list of
    per-batch tuples. Measured on transformers 5.14.1, each tuple has 14 entries
    and entry 0 is the LOSS DICT, not the logits:

        predictions[i][0] -> dict of loss terms
        predictions[i][1] -> ndarray (B, 300, num_labels)
        predictions[i][2] -> ndarray (B, 300, 4)

    Indexing positionally is what broke the first Colab run - `batch[0]` returned
    the loss dict and torch.as_tensor raised "Could not infer dtype of dict".
    Selecting by trailing dimension survives the tuple layout changing again,
    which it already has once.
    """

    import numpy as np

    if isinstance(batch, dict):
        candidates = list(batch.values())
    elif isinstance(batch, (list, tuple)):
        candidates = list(batch)
    else:
        candidates = [batch]

    logits = boxes = None
    for item in candidates:
        if not hasattr(item, "shape") or getattr(item, "ndim", 0) != 3:
            continue
        array = np.asarray(item)
        if logits is None and array.shape[-1] == num_labels:
            logits = array
        elif boxes is None and array.shape[-1] == 4:
            boxes = array

    if logits is None or boxes is None:
        shapes = [
            tuple(getattr(item, "shape", ())) if hasattr(item, "shape") else type(item).__name__
            for item in candidates
        ]
        raise EvalStructureError(
            f"Could not locate logits (..., {num_labels}) and boxes (..., 4) in the "
            f"eval batch. Saw: {shapes}"
        )
    return logits, boxes


def predictions_to_coco(
    processed: Sequence[dict[str, Any]], image_ids: Sequence[int]
) -> list[dict[str, Any]]:
    """Convert post_process_object_detection output into COCO detections.

    post_process returns xyxy in the original image scale; COCO wants xywh.
    """

    detections: list[dict[str, Any]] = []
    for result, image_id in zip(processed, image_ids, strict=True):
        boxes = result["boxes"].detach().cpu().numpy()
        scores = result["scores"].detach().cpu().numpy()
        labels = result["labels"].detach().cpu().numpy()
        for (x0, y0, x1, y1), score, label in zip(boxes, scores, labels, strict=True):
            width, height = float(x1) - float(x0), float(y1) - float(y0)
            if width <= 0 or height <= 0:
                continue
            detections.append(
                {
                    "image_id": int(image_id),
                    "category_id": int(label),
                    "bbox": [float(x0), float(y0), width, height],
                    "score": float(score),
                }
            )
    return detections
'''
MODULE_SOURCE['run'] = r'''"""One arm, end to end. Shared by the local smoke test and the Colab notebook.

TRAIN-13 asks for a local 1-step smoke test, which is only meaningful if the
smoke test exercises the same code path Colab will. So the notebook calls this
function too, with different paths and a real step budget.
"""

from __future__ import annotations

import json
import os
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch

from src.training.arms import ArmComposition
from src.training.data import (
    DetectionDataset,
    assert_eval_batch_remainder_is_safe,
    build_transform,
    collate,
    load_coco_samples,
)
from src.training.metrics import (
    build_coco_ground_truth,
    evaluate_detections,
    extract_logits_and_boxes,
    predictions_to_coco,
)
from src.training.trainer import (
    RTDetrTrainer,
    build_training_arguments,
    find_resumable_checkpoint,
)

CLASS_NAMES = ("helmet", "head", "person")


@dataclass(frozen=True)
class RunPaths:
    real_images: Path
    real_coco: Path
    synthetic_images: Path
    synthetic_coco: Path | None
    output_dir: Path


def load_model_and_processor(checkpoint: str, *, dtype=torch.float32):
    """Auto classes only. ADR-014: the named RTDetrV2ImageProcessor does not exist."""

    from transformers import AutoImageProcessor, AutoModelForObjectDetection

    processor = AutoImageProcessor.from_pretrained(checkpoint)
    model = AutoModelForObjectDetection.from_pretrained(
        checkpoint,
        num_labels=len(CLASS_NAMES),
        # Mandatory: the 80-class head cannot be reused for 3 classes.
        ignore_mismatched_sizes=True,
        id2label=dict(enumerate(CLASS_NAMES)),
        label2id={name: index for index, name in enumerate(CLASS_NAMES)},
        dtype=dtype,
    )
    return model, processor


def build_datasets(
    composition: ArmComposition,
    paths: RunPaths,
    processor,
    config: Mapping[str, Any],
):
    train_samples = load_coco_samples(
        paths.real_coco,
        paths.real_images,
        keep_names=composition.real_train,
    )
    if composition.synthetic and paths.synthetic_coco is not None:
        train_samples += load_coco_samples(
            paths.synthetic_coco,
            paths.synthetic_images,
            keep_names=composition.synthetic,
            is_synthetic=True,
        )
    # TRAIN-23: validation is real only, always.
    val_samples = load_coco_samples(
        paths.real_coco, paths.real_images, keep_names=composition.real_val
    )

    transform = build_transform(composition.augmentation_profile, config)
    return (
        DetectionDataset(train_samples, processor, transform),
        DetectionDataset(val_samples, processor, None),
        val_samples,
    )


def make_compute_metrics(processor, val_samples):
    """COCOeval over the whole val set; AP_small is a headline metric."""

    ground_truth = build_coco_ground_truth(val_samples, CLASS_NAMES)
    image_ids = [int(sample.image_id) for sample in val_samples]
    # (height, width) per image, in the order the eval dataloader yields them.
    # Trainer's eval loader does not shuffle, so this order is the dataset order.
    target_sizes_all = [[sample.height, sample.width] for sample in val_samples]

    def compute_metrics(eval_prediction) -> dict[str, float]:
        from types import SimpleNamespace

        predictions = eval_prediction.predictions
        # eval_do_concat_batches=False means predictions arrive as a list of
        # per-batch outputs rather than one concatenated array.
        batches = (
            predictions
            if isinstance(predictions, list)
            else [predictions]
        )
        logits_batches, boxes_batches = [], []
        for batch in batches:
            logits, boxes = extract_logits_and_boxes(
                batch, num_labels=len(CLASS_NAMES)
            )
            logits_batches.append(torch.as_tensor(logits))
            boxes_batches.append(torch.as_tensor(boxes))
        logits = torch.cat(logits_batches, dim=0).float()
        boxes = torch.cat(boxes_batches, dim=0).float()

        count = int(logits.shape[0])
        outputs = SimpleNamespace(logits=logits, pred_boxes=boxes)
        target_sizes = torch.tensor(target_sizes_all[:count], dtype=torch.float32)
        processed = processor.post_process_object_detection(
            outputs, threshold=0.0, target_sizes=target_sizes
        )
        detections = predictions_to_coco(processed, image_ids[:count])
        return evaluate_detections(ground_truth, detections)

    return compute_metrics


def run_arm(
    composition: ArmComposition,
    paths: RunPaths,
    *,
    config: Mapping[str, Any],
    total_steps: int,
    seed: int,
    resume: bool = True,
) -> dict[str, Any]:
    """Train one arm, resuming automatically if a checkpoint is present."""

    run_config = config["run"]
    eval_batch = int(run_config["per_device_eval_batch_size"])
    assert_eval_batch_remainder_is_safe(len(composition.real_val), eval_batch)

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model, processor = load_model_and_processor(config["model"]["checkpoint"])
    train_dataset, val_dataset, val_samples = build_datasets(
        composition, paths, processor, config
    )

    workers = int(
        run_config["dataloader_num_workers_colab"]
        if os.environ.get("COLAB_GPU") or Path("/content").exists()
        else run_config["dataloader_num_workers_windows"]
    )
    arguments = build_training_arguments(
        output_dir=str(paths.output_dir),
        config=config,
        total_steps=total_steps,
        seed=seed,
        use_bf16=use_bf16,
        dataloader_num_workers=workers,
    )

    trainer = RTDetrTrainer(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate,
        compute_metrics=make_compute_metrics(processor, val_samples),
        optimizer_config=config["optimizer"],
    )

    checkpoint = find_resumable_checkpoint(paths.output_dir) if resume else None
    result = trainer.train(resume_from_checkpoint=checkpoint)
    # Evaluate explicitly at the end so the record always carries metrics, even
    # when eval_strategy would not have fired on the final step.
    final_metrics = trainer.evaluate()

    record = {
        "arm": composition.arm,
        "seed": seed,
        "total_steps": total_steps,
        "resumed_from": checkpoint,
        "train_runtime_seconds": float(result.metrics.get("train_runtime", 0.0)),
        "train_loss": float(result.metrics.get("train_loss", float("nan"))),
        "eval_metrics": {
            key: float(value)
            for key, value in final_metrics.items()
            if isinstance(value, (int, float))
        },
        "composition": composition.summary(),
        "bf16": bool(use_bf16),
    }
    paths.output_dir.mkdir(parents=True, exist_ok=True)
    (paths.output_dir / "run_record.json").write_text(
        json.dumps(record, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    return record
'''

for _name, _source in MODULE_SOURCE.items():
    (PKG / f"{_name}.py").write_text(_source, encoding="utf-8")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("training modules written:", sorted(MODULE_SOURCE))

## 5. 組成四組並檢查不變式

In [ ]:
import json
import pathlib

import yaml

from src.training.arms import build_all_arms, equal_step_budget

DATA = pathlib.Path("/content/data")
config = yaml.safe_load(
    (DATA / "configs" / "training.yaml").read_text(encoding="utf-8")
)

arms = build_all_arms(
    manifest_path=DATA / "splits" / "split_manifest.json",
    synthetic_annotations={
        "filtered": DATA / "synthetic" / "annotations_filtered_1x.json",
        "unfiltered": DATA / "synthetic" / "annotations_unfiltered_1x.json",
    },
)
print("不變式通過：TRAIN-04 相同真實影像 / TRAIN-06 合成等量 / TRAIN-23 無 Test")
for _arm, _comp in arms.items():
    _s = _comp.summary()
    print(
        f"  {_arm:<16} real={_s['n_real_train']} + syn={_s['n_synthetic']}"
        f" = {_s['n_train_total']}  aug={_s['augmentation_profile']}"
    )

BATCH = int(config["run"]["per_device_train_batch_size"])
plan = equal_step_budget(
    arms,
    reference_arm="real_only",
    reference_epochs=int(config["run"]["num_train_epochs_real_only"]),
    batch_size=BATCH,
)
print()
for _arm, _row in plan.items():
    print(
        f"  {_arm:<16} steps={_row['total_steps']}  epochs={_row['epochs']:.1f}"
        f"  每張真實圖看 {_row['real_image_exposures']:.1f} 次"
    )

## 6. 依序訓練四組

每組一個獨立輸出目錄（TRAIN-11），每組跑完立刻同步回 Drive（TRAIN-09）。
**斷線後重新執行這格會自動接續**（TRAIN-10）。

In [ ]:
import shutil
import time
import traceback

from src.training.run import RunPaths, run_arm

SEED = int(config["seeds"]["primary"])
RUNS = pathlib.Path("/content/runs")
DRIVE_RUNS = pathlib.Path(DRIVE_DIR) / "runs"
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

SUBSET = {"unfiltered_syn": "unfiltered", "filtered_syn": "filtered"}
records = []

for arm in ("real_only", "standard_aug", "unfiltered_syn", "filtered_syn"):
    output_dir = RUNS / arm / f"seed_{SEED}"
    drive_dir = DRIVE_RUNS / arm / f"seed_{SEED}"

    # Bring a previous session's checkpoints back before deciding to resume.
    if drive_dir.is_dir() and not output_dir.is_dir():
        shutil.copytree(drive_dir, output_dir)
        print(f"[{arm}] restored checkpoints from Drive")

    subset = SUBSET.get(arm)
    run_paths = RunPaths(
        real_images=DATA / "real" / "images",
        real_coco=DATA / "real" / "coco_all.json",
        synthetic_images=DATA / "synthetic" / "images",
        synthetic_coco=(
            DATA / "synthetic" / f"annotations_{subset}_1x.json" if subset else None
        ),
        output_dir=output_dir,
    )

    print(f"\n===== {arm} =====")
    started = time.time()
    try:
        record = run_arm(
            arms[arm],
            run_paths,
            config=config,
            total_steps=plan[arm]["total_steps"],
            seed=SEED,
            resume=True,
        )
    except Exception:  # noqa: BLE001 - one failed arm must not stop the other three
        traceback.print_exc()
        print(f"[{arm}] FAILED — 其餘組別繼續，稍後單獨重跑這一組即可")
        continue

    record["wall_clock_hours"] = round((time.time() - started) / 3600, 3)
    records.append(record)
    print(f"[{arm}] done in {record['wall_clock_hours']:.2f} h")

    # Sync after each arm, not at the end: a disconnect in hour four must not
    # cost the three arms that already finished.
    drive_dir.parent.mkdir(parents=True, exist_ok=True)
    if drive_dir.is_dir():
        shutil.rmtree(drive_dir)
    shutil.copytree(output_dir, drive_dir)
    print(f"[{arm}] synced to Drive")

print("\n完成的組別:", [r["arm"] for r in records])

## 7. 打包結果回 Drive

跑完把 Drive 上的 `results_colab.zip` 下載回本機，放進 repo 的 `results/colab/`。

In [ ]:
summary = {
    "records": records,
    "plan": plan,
    "arms": {a: c.summary() for a, c in arms.items()},
    "gpu": torch.cuda.get_device_name(0),
    "bf16": torch.cuda.is_bf16_supported(),
    "transformers": transformers.__version__,
}

out = pathlib.Path("/content/results_colab")
out.mkdir(exist_ok=True)
(out / "training_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True, default=str), encoding="utf-8"
)
for arm_dir in RUNS.glob("*/seed_*"):
    for name in ("run_record.json", "trainer_state.json"):
        source = arm_dir / name
        if source.is_file():
            target = out / arm_dir.parent.name / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)

archive_path = shutil.make_archive("/content/results_colab", "zip", str(out))
shutil.copy2(archive_path, pathlib.Path(DRIVE_DIR) / "results_colab.zip")
print("寫到 Drive:", pathlib.Path(DRIVE_DIR) / "results_colab.zip")
print(json.dumps(summary["records"], indent=2, default=str))